In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


# Izhikevich神经元模型的更新函数
def izhikevich_update(v, u, a, b, c, d, I, g=0.0, v_other=None):
    """
    更新Izhikevich神经元的状态变量 v 和 u
    :param v: 当前膜电位
    :param u: 当前恢复变量
    :param a: 模型参数
    :param b: 模型参数
    :param c: 模型参数
    :param d: 模型参数
    :param I: 外部输入电流
    :param g: 耦合强度
    :param v_other: 与之耦合的另一个神经元的膜电位，默认为None（无耦合）
    :return: 更新后的 v 和 u
    """
    dv = 0.04 * v ** 2 + 5 * v + 140 - u + I
    if v_other is not None:
        dv += g * (v_other - v)
    du = a * (b * v - u)
    v += dv * 0.5
    u += du * 0.5
    if v >= 30:
        v = c
        u += d
    return v, u


# 计算Kuramoto序参量，用于衡量同步性
def kuramoto_order_parameter(phases):
    """
    计算Kuramoto序参量
    :param phases: 神经元的相位列表
    :return: Kuramoto序参量的值
    """
    r = np.sum(np.exp(1j * phases)) / len(phases)
    return np.abs(r)


# 模拟神经元网络
def simulate_neuron_network(G, T, dt, params, I, g, W):
    """
    模拟有向小世界网络中的神经元网络行为
    :param G: 网络结构
    :param T: 总时间
    :param dt: 时间步长
    :param params: 包含每个神经元的参数的字典
    :param I: 外部输入电流
    :param g: 耦合强度
    :param W: 传播权重矩阵
    :return: 神经元的膜电位和相位信息
    """
    N = len(G.nodes())
    v = np.full(N, -65.0, dtype=np.float64)
    u = np.full(N, 0.0, dtype=np.float64)
    v[0] = 60
    v_hist = []
    phases = []
    steps = int(T / dt)
    for _ in range(steps):
        v_list = []
        for i in G.nodes():
            v_neighbors = []
            for j in G.predecessors(i):
                v_neighbors.append(v[j] * W[j, i])
            v_other = np.sum(v_neighbors) if v_neighbors else None
            a, b, c, d = params["a"][i], params["b"][i], params["c"][i], params["d"][i]
            v[i], u[i] = izhikevich_update(v[i], u[i], a, b, c, d, I[i], g, v_other)
            v_list.append(v[i])
        v_hist.append(v_list)
        phases.append([np.angle(np.exp(1j * v)) for v in v_list])
    return np.array(v_hist), np.array(phases)


# 假设的 MSF 计算函数，需要根据实际模型进行调整
def msf(alpha, beta):
    """
    计算主稳定性函数，这里只是一个示例函数，实际的 MSF 计算会更复杂
    :param alpha: 耦合参数 1
    :param beta: 耦合参数 2
    :return: MSF 值
    """
    return np.sin(alpha) * np.cos(beta)


# 计算并绘制 MSF 等高线图
def plot_msf():
    """
    计算并绘制 MSF 等高线图
    """
    alpha = np.linspace(-1.5, 0.1, 100)
    beta = np.linspace(-5, 0.1, 100)
    alpha, beta = np.meshgrid(alpha, beta)
    lambda_max = msf(alpha, beta)
    fig, ax = plt.subplots()
    contour = ax.contour(alpha, beta, lambda_max, levels=20, cmap='viridis')
    ax.clabel(contour, inline=True, fontsize=8)
    ax.set_xlabel(r'$\alpha$')
    ax.set_ylabel(r'$\beta$')
    ax.set_title('Contour Map of Master Stability Function')
    # 保存图片
    plt.savefig('msf_contour_plot.png')
    plt.show()


# 创建有向小世界网络
def create_directed_small_world_network():
    """
    创建有向小世界网络并设置初始条件
    """
    N = 5000  # 节点数量
    k = 3  # 每个节点的固定出向边数量
    G = nx.DiGraph()
    G.add_nodes_from(range(N))
    np.random.seed(42)
    # 确保每个节点有6个唯一的出向边
    for node in range(N):
        targets = np.random.choice([n for n in range(N) if n!= node], k, replace=False)
        for target in targets:
            G.add_edge(node, target)
    # 为每个节点分配三维坐标，使得相邻节点尽可能靠近
    layout = nx.spring_layout(G, dim=3, seed=42)  # 使用 spring layout 生成 3D 坐标
    pos = {i: layout[i] for i in range(N)}
    # 初始化膜电位和恢复变量
    V = np.full(N, -65.0, dtype=np.float64)  # 初始膜电位为-65 mV
    V[0] = 60
    u = np.full(N, 0.0, dtype=np.float64)  # 初始恢复变量为0
    # 随机化每个神经元的模型参数
    params = {
        "a": np.random.uniform(0.01, 0.03, N),
        "b": np.random.uniform(0.1, 0.3, N),
        "c": np.random.uniform(-68, -58, N),
        "d": np.random.uniform(2, 10, N),
    }
    # 设置随机输入，初始仅对一个节点施加强刺激
    I = np.zeros(N)
    I[0] = 150  # 增强初始输入强度，确保刺激传播
    # 创建传播权重矩阵（固定随机权重）
    W = np.zeros((N, N))
    for i, j in G.edges():
        W[i, j] = np.random.uniform(0.8, 1.2)  # 权重在 0.8 到 1.2 之间波动
    return G, params, I, W


# 主程序
if __name__ == "__main__":
    G, params, I, W = create_directed_small_world_network()
    T = 1000  # 总时间
    dt = 0.1  # 时间步长
    g = 0.5  # 初始耦合强度，可根据需要调整
    _, _ = simulate_neuron_network(G, T, dt, params, I, g, W)
    plot_msf()